In [73]:
import numpy as np
from amuse.ic.kroupa import new_kroupa_mass_distribution
from amuse.units import units
from amuse.ic.salpeter import new_salpeter_mass_distribution
from amuse.lab import nbody_system
from amuse.lab import *
from amuse.ext.orbital_elements import new_binary_from_orbital_elements
from amuse.plot import scatter
from amuse.ext.orbital_elements import get_orbital_elements_from_binary
from amuse.community.ph4.interface import Ph4
from amuse.ext.orbital_elements import semimajor_axis_to_orbital_period
from amuse.units.core import named_unit
from amuse.units import units

## Derivations for parameter sweep:
Begin with full energy and angular momentum expressions (in COM frame)
### $E = \frac{1}{2}m_1v_1^2 + \frac{1}{2}m_2v_2^2 - \frac{Gm_1m_2}{|r_1 - r_2|}$
### $L = m_1(r_1 \times v_1) + m_2(r_2 \times v_2)$

also the COM frame implies:
### $m_1 r_1 + m_2 r_2 = 0$
### $m_1 v_1 + m_2 v_2 = 0$

reduction to one body:
### $r = r_1 - r_2$
### $v = v_1 - v_2$
### $M = m_1 + m_2$
### $\mu = \frac{m_1 m_2}{m_1+m_2}$
inserting which  into the COM conditions gives:
### $r_1 = \frac{m_2 r}{M}$, $r_2 = -\frac{m_1 r}{M}$
### $v_1 = \frac{m_2 v}{M}$, $v_2 = -\frac{m_1 v}{M}$
insert these into E and L expressions to obtain an equivalent one-body problem with mass $\mu$:
### $E = \frac{1}{2}\mu v^2 - \frac{Gm_1 m_2}{r}$ 
### $L = \mu (r \times v)$
now use Keplerian orbit formulation
### $v^2 = \dot{r}^2 + r^2\dot{\theta}^2$ 
### $L = \mu r^2 \dot{\theta} => \dot{\theta} = \frac{L}{\mu r^2}$
substitute $v^2$ into energy equation:
### $E = \frac{1}{2} \mu \dot{r}^2 + \frac{L^2}{2\mu r^2}-\frac{Gm_1 m_2}{r}$
at pericentre $\dot{r} = 0$ =>

# $E = \frac{L^2}{2\mu r_p^2}-\frac{Gm_1 m_2}{r_p}$

then substitute the orbit (conic section) equation
### $r(f) = \frac{p}{1+e \ cos f}$, where f is true anomaly
into the above energy equation to get the eccentricity:

# $e = \sqrt{1+\frac{2EL^2}{\mu^3 G^2 (m_1+m_2)^2}}$
again from the conic section equation, setting r=r_p and inserting eccentricity we get:
# $r_p = \frac{L^2}{\mu G m_1 m_2 (1+e)}$



so, for the sweep we need:
### $m_1, m_2, E, L$ => $e, r_p$

to initialize initial conditions, we again use the relative seperation:
### $r(f) = \frac{p}{1+e \ cos f}$, where f is true anomaly and p:
### $p = \frac{L^2}{\mu G (m_1 + m_2)}$

now, $r$ and $v$ vectors are expressed as:
### $r = r(f) \ [(cos f)_x \ (sin f)_y]$
### $v = \frac{G(m_1 + m_2) \mu}{L} \ [(-sin f)_x \ (e+cos f)_y]$

and from the previous expressions we already have:
### $r_1 = \frac{m_2 r}{M}$, $r_2 = -\frac{m_1 r}{M}$
### $v_1 = \frac{m_2 v}{M}$, $v_2 = -\frac{m_1 v}{M}$

to initialize one simulation we can choose an r value (that we keep the same for all cases)
from the r value we find the true anomaly f and thus the initial r & v values!

In [74]:
MBlackhole = named_unit("70 solar masses", "MBH70", 70 * units.MSun)

In [75]:
from amuse.units import units, constants 
import numpy as np
#These are hypothetical values for the system (will change accordingly)
m1 = 10 | units.MSun                    # original black hole
m2 = 1 | units.MSun                     # original companion star
m3 = 10 | units.MSun                    # incoming black hole
b = 1.5 | units.AU                       # impact parameter of the incoming black hole*                
v_inf = 20 | units.kms                     # relative velocity of the incoming black hole*

M12 = m1 + m2
M_total = M12 +m3
mu_out = (M12 * m3) / M_total

#Incoming orbit relative to the original binary's center of mass
E_out = 0.5 * mu_out * v_inf**2
L_out = mu_out * b * v_inf

#Keplerian outer-orbit parameters
e_out = np.sqrt(1 + (2 * E_out * L_out**2) / (mu_out ** 3 *(constants.G * M_total)**2))
p_out = L_out**2 / (mu_out * constants.G * M_total)
r_peri = p_out / (1 + e_out)
print("Outer-orbit energy", E_out)
print("Outer angular momentum", L_out)
print("Outer-orbit eccentricity", e_out)
print("Pericenter:", r_peri)


Outer-orbit energy 1047.6190476190477 1.9889200000000003e+36 * m**2 * kg * s**-2
Outer angular momentum 157.14285714285717 2.9753819697474374e+44 * m**2 * kg * s**-1
Outer-orbit eccentricity 1.00051823739
Pericenter: 1.68131724683e+12 2.237952291528116e+28 * m * kg


In [76]:
def get_orb_params(m1, m2, E = -3.0E39 | units.J, L = 1.3E49 | units.kg * units.m**2 / units.s, initial_seperation = 200 | units.AU):
    """given masses of the 2 objects (m2 being the 'incoming' object), total energy, total angular
    momentum and the initial seperation, calculates corresponding eccentricity and true anomaly"""
    
    #needed values for calculation (reduced mass & semi latus rectum)
    total_mass = m1 + m2
    red_mass = (m1*m2) / (total_mass)
    mu = constants.G * total_mass 
    p = (L**2) / (red_mass**2 * mu) #semi latus rectum

    #calculate eccentricity of the orbit
    eccentricity = np.sqrt(1 + ((2 * E * L**2) / ((red_mass**3) * (mu**2))))
    eccentricity_squared = eccentricity**2
    eccentricity_square_val = float(eccentricity_squared.value_in(units.none) if hasattr(eccentricity_squared, "unit") else eccentricity_squared)
    eccentricity = np.sqrt(max(0.0, eccentricity_square_val))

    #given eccentricity and p (semi latus rectum) we can then get the true anomaly corresponding to the initial seperation
    #we need this true anomaly to get the initial conditions for the run
    if eccentricity == 0.0:
        true_anom_rad = 0.0
    else:
        cos_true_anom = (p - initial_seperation) / (initial_seperation * eccentricity)
        cos_true_anom_val = float(cos_true_anom.value_in(units.none) if hasattr(cos_true_anom, "unit") else cos_true_anom)
        cos_true_anom_clipped = np.clip(cos_true_anom_val, -1.0, 1.0)

        true_anom_rad = np.arccos(cos_true_anom_clipped)

    phase_out = float(np.degrees(true_anom_rad)) | units.deg

    return eccentricity, true_anom_rad, phase_out


def get_initial_conditions(true_anom, eccentricity, m1, m2, L):
    """given the true anomaly and eccentricity of the orbit, and the two masses and total angular momentum
    calculates the initial conditions in the COM frame (v1, r1, v2, r2)"""

    #needed values for calculation (reduced mass, total mass & semi latus rectum)
    red_mass = m1*m2/(m1+m2)
    totmass = m1+m2
    p = (L**2)/(red_mass*totmass*constants.G)

    #COM v and r values
    r_com = p/(1+eccentricity*np.cos(true_anom))
    v_com = (constants.G*red_mass*totmass)/L

    r_com_x = r_com*np.cos(true_anom)
    r_com_y = r_com*np.sin(true_anom)
    v_com_x = v_com*(-np.sin(true_anom))
    v_com_y = v_com*(eccentricity+np.cos(true_anom))

    #now converting these to r1, v1, r2, v2
    r1_x = r_com_x*m2/totmass
    r1_y = r_com_y*m2/totmass
    r2_x = -r_com_x*m1/totmass
    r2_y = -r_com_y*m1/totmass

    v1_x = v_com_x*m2/totmass
    v1_y = v_com_y*m2/totmass
    v2_x = -v_com_x*m1/totmass
    v2_y = -v_com_y*m1/totmass

    return r1_x, r1_y, v1_x, v1_y, r2_x, r2_y, v2_x, v2_y

In [77]:
array_of_energies = np.random.rand(10) #size of 10 t0 start
array_of_angular_momenta = np.random.rand(10) #size of 10 to start

eccentricities = []
true_anomalies = []
for i in array_of_energies:
    for j in array_of_angular_momenta: 
        eccentricity, true_anom, _ = get_orb_params(m1, m2, i, j, initial_seperation)
        eccentricities.append(eccentricity)
        true_anomalies.append(true_anom)
eccentricities = np.array(eccentricities)
true_anomalies = np.array(true_anomalies)
for idx in range(len(eccentricities)):
    i = eccentricities[idx]
    j = true_anomalies[idx]
    r1_x, r1_y, v1_x, v1_y, r2_x, r2_y, v2_x, v2_y = get_initial_conditions(j, i, m1, m2, j)
        

NameError: name 'initial_seperation' is not defined

In [78]:
#for a 2 body system

def initial_conditions_blackhole_star_system():
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    semimajor_axis_blackhole_2 = 300 | units.AU
    eccentricity = eccentricity
    pericenter = pericenter

    # star_blackhole_binary = new_binary_from_orbital_elements( #inner binary
    #     mass1=mass_blackhole_1,
    #     mass2=mass_star,
    #     semimajor_axis=semimajor_axis_star,
    #     eccentricity=eccentricity,
    #     G=constants.G
    # )
    # star_blackhole_binary.name = ['Black Hole 1', 'Star']

    blackhole_binary = new_binary_from_orbital_elements( #outer binary
        mass1 = mass_blackhole_1, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = semimajor_axis_blackhole_2, 
        eccentricity = eccentricity, 
        pericenter = pericenter,
        G=constants.G 
    )
    blackhole_binary.name = ['Fake Black Hole 1 (do not use)', 'Black Hole 2']

    # Shift the inner binary (star and black hole 1) to match the position of the fake black hole 1 in the outer binary
    fake_black_hole_1 = blackhole_binary[blackhole_binary.name == 'Fake Black Hole 1 (do not use)']
    shift_pos = fake_black_hole_1.position - star_blackhole_binary[0].position
    shift_vel = fake_black_hole_1.velocity - star_blackhole_binary[0].velocity
    star_blackhole_binary.position += shift_pos
    star_blackhole_binary.velocity += shift_vel

    #position black hole 2 relative to the orter binary orbit
    blackhole_2 = blackhole_binary[blackhole_binary.name == 'Black Hole 2'].copy()
    blackhole_2.position = star_blackhole_binary[0].position + (blackhole_binary[1].position - blackhole_binary[0].position)
    blackhole_2.velocity = star_blackhole_binary[0].velocity + (blackhole_binary[1].velocity - blackhole_binary[0].velocity)
    blackhole_2.name = ['Black Hole 2']
    
    star_blackhole_binary.add_particles(blackhole_2)
    star_both_blackholes = star_blackhole_binary.copy()
    return star_both_blackholes

if __name__ == "__main__":
    for idx, e_out in enumerate(eccentricities):
        for jdx, r_peri in enumerate(pericenters):
            star_both_blackholes = initial_conditions_blackhole_star_system(
                eccentricity_out = e_out, 
                pericenter_out = r_peri,
                eccentricity_star=0.0, 
                semimajor_axis_star=30|units.AU
                )
            
            a = [] | units.au
            e = []
    

            orbit_inner = get_orbital_elements_from_binary(star_both_blackholes[0:2], G = constants.G)
            print(f'black hole 1 and star orbit: {orbit_inner}')
            a.append(orbit_inner[2])
            e.append(orbit_inner[3])


            combined_inner = Particles(1) #star and black hole 1 together
            combined_inner.mass = star_both_blackholes[0:2].mass.sum()
            combined_inner.position = star_both_blackholes[0:2].center_of_mass()
            combined_inner.velocity = star_both_blackholes[0:2].center_of_mass_velocity()

            outer_system = Particles(2) #combined inner binary and black hole 2
            outer_system.add_particle(combined_inner[0])
            outer_system.add_particle(star_both_blackholes[2])
            
            orbit_outer = get_orbital_elements_from_binary(outer_system, G = constants.G)
            print(f'black hole 1/star and black hole 2 orbit: {orbit_outer}')
            a.append(orbit_outer[2])
            e.append(orbit_outer[3])

            scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()), c='b', marker="*")

            scatter(a, e)

            converter = nbody_system.nbody_to_si(star_both_blackholes.mass.sum(), star_both_blackholes[0].position.length())
            gravity = Ph4(converter)
            gravity.parameters.timestep_parameter = 0.01
            gravity.particles.add_particles(star_both_blackholes)
            print(f'gravity.particles: {gravity.particles}')
            print(f'bodies (star and bh1 and bh2): {star_both_blackholes}')

            channel_from_gravity = gravity.particles.new_channel_to(star_both_blackholes)
            channel_from_gravity.copy()
            print(f'updated bodies (star and bh1 and bh2): {star_both_blackholes}')

            Porb = semimajor_axis_to_orbital_period(a[0], star_both_blackholes[0:2].mass.sum())
            print(f'Orbital period of black hole 1 and star: {Porb}')

            Porb2 = semimajor_axis_to_orbital_period(a[1], star_both_blackholes[0:2].mass.sum() + star_both_blackholes[2].mass)
            print(f'Orbital period of inner binary and black hole 2: {Porb2}')

            Ek0 = star_both_blackholes.kinetic_energy()
            Ep0 = star_both_blackholes.potential_energy()
            Etot0 = Ek0 + Ep0

            time = 0 | units.yr
            dt = Porb / 100
            index = 0
            while gravity.model_time < Porb:
                time += dt
                gravity.evolve_model(time)
                channel_from_gravity.copy()

                Ek = star_both_blackholes.kinetic_energy()
                Ep = star_both_blackholes.potential_energy()
                Etot = Ek + Ep
                
                print(f"time={time.in_(units.yr)}, dE = {(Etot-Etot0)/Etot0}")

                plt.xlim(-500, 500)
                plt.ylim(-500, 500)
                scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()))
                index += 1        
                filename = f"binary_snapshot_e{idx}_p{jdx}_i{index:04}.jpg"
                plt.savefig( filename)
                plt.close()
            gravity.stop()

In [79]:
#this is for a 3-body system

def initial_conditions_blackhole_star_system(
        eccentricity_out, 
        eccentricity_star=0.0, 
        semimajor_axis_star=30|units.AU, 
        semimajor_axis_outer=300|units.AU,
        inclination_out = 0.0 | units.deg,
        true_anomaly_out = 0.0 | units.deg,
        true_anomaly_star = 0.0 | units.deg
        ):
   
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    mass_star = 1/70 | MBlackhole
    semimajor_axis_star = (semimajor_axis_star if hasattr(semimajor_axis_star, 'unit') else semimajor_axis_star | units.AU)
    eccentricity_star = eccentricity_star
    semimajor_axis_outer = (semimajor_axis_outer if hasattr(semimajor_axis_outer, 'unit') else semimajor_axis_outer | units.AU)
    eccentricity = eccentricity_out
    true_anomaly = true_anomaly_out if hasattr(true_anomaly_out, 'unit') else true_anomaly_out | units.deg
    true_anomaly_star = true_anomaly_star if hasattr(true_anomaly_star, 'unit') else true_anomaly_star | units.deg
    inclination = inclination_out if hasattr(inclination_out, 'unit') else inclination_out | units.deg

    star_blackhole_binary = new_binary_from_orbital_elements( #inner binary
        mass1=mass_blackhole_1,
        mass2=mass_star,
        semimajor_axis=semimajor_axis_star,
        eccentricity=eccentricity_star,
        true_anomaly=true_anomaly_star,
        G=constants.G
    )
    star_blackhole_binary.name = ['Black Hole 1', 'Star']

    inner_binary_mass = mass_blackhole_1 + mass_star
    total_mass = inner_binary_mass + mass_blackhole_2

    blackhole_binary = new_binary_from_orbital_elements( #outer binary
        mass1 = inner_binary_mass, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = semimajor_axis_outer, 
        eccentricity = eccentricity, 
        true_anomaly = true_anomaly,
        inclination = inclination,
        G=constants.G 
    )
    blackhole_binary.name = ['Fake Black Hole 1 (do not use)', 'Black Hole 2']

    # Shift the inner binary (star and black hole 1) center of mass to match the position of the fake black hole 1 in the outer binary
    fake_black_hole_1 = blackhole_binary[blackhole_binary.name == 'Fake Black Hole 1 (do not use)'][0]
    com_pos = star_blackhole_binary.center_of_mass()
    com_vel = star_blackhole_binary.center_of_mass_velocity()
    shift_pos = fake_black_hole_1.position - com_pos
    shift_vel = fake_black_hole_1.velocity - com_vel
    star_blackhole_binary.position += shift_pos
    star_blackhole_binary.velocity += shift_vel

    #position black hole 2 relative to the orter binary orbit
    blackhole_2 = blackhole_binary[blackhole_binary.name == 'Black Hole 2'].copy()
    blackhole_2.position = star_blackhole_binary.center_of_mass() + (blackhole_binary[1].position - blackhole_binary[0].position)
    blackhole_2.velocity = star_blackhole_binary.center_of_mass_velocity() + (blackhole_binary[1].velocity - blackhole_binary[0].velocity)
    blackhole_2.name = ['Black Hole 2']
    
    star_blackhole_binary.add_particles(blackhole_2)
    star_both_blackholes = star_blackhole_binary.copy()
    star_both_blackholes.position -= star_both_blackholes.center_of_mass()
    star_both_blackholes.velocity -= star_both_blackholes.center_of_mass_velocity()
    return star_both_blackholes

In [80]:
eccentricities = [0.0, 0.1, 0.3]
semimajor_axes = [30 | units.AU, 50 | units.AU, 100 | units.AU]
inclinations = [0.0, 30.0, 60.0] | units.deg
phases_out = [0.0, 90.0, 180.0] | units.deg

In [81]:
import os

os.environ["OMPI_MCA_btl_tcp_if_include"] = "127.0.0.1/8"
os.environ["OMPI_MCA_oob_tcp_if_include"] = "127.0.0.1/8"

In [82]:
def get_inclinations(mode = "kozai_regimes", n_inc=3, max_deg=90.0, seed=None):
    """
    Generates outer inclination angles from physical criteria.
    The different modes are:
      - 'kozai_regimes': coplanar (0 deg), Kozai critical (~39.23 deg), polar (90 deg)
      - 'kozai_full': full range including retrograde Kozai (~140.77 deg) and retrograde coplanar (180 deg)
      - 'isotropic_grid': deterministic grid uniform in cos(i) from 0 deg to max_deg
      - 'isotropic_random': random sample uniform in cos(i) from 0 deg to max_deg
    """
    kozai_crit_deg = float(np.degrees(np.arccos(np.sqrt(3.0 / 5.0)))) # Kozai critical angle in degrees

    if mode == "kozai_regimes":
        vals = [0.0, kozai_crit_deg, 90.0][:n_inc] #three key prograde regimes—coplanar (0.0°), the Kozai–Lidov threshold (~39.23°), and polar (90.0°). slices [:n_inc] in case you ask for fewer than 3 angles
    elif mode == "kozai_full":
        kozai_retro_deg = 180.0 - kozai_crit_deg
        vals = [0.0, kozai_crit_deg, 90.0, kozai_retro_deg, 180.0][:n_inc] #5-point list spanning prograde coplanar (0°), prograde Kozai critical (~39.23°), polar (90°), retrograde Kozai critical (~140.77°), and retrograde coplanar (180°)
    elif mode == "isotropic_grid":
        cos_start = 1.0 # cos(0°), starting point for isotropic grid
        cos_end = float(np.cos(np.radians(max_deg))) # cos(max_deg), ending point for isotropic grid
        cos_vals = np.linspace(cos_start, cos_end, n_inc) # generate n_inc evenly spaced values in cos(i) for the isotropic grid
        vals = np.degrees(np.arccos(cos_vals)) # convert the isotropic grid cos(i) values to inclination angles in degrees
    elif mode == "isotropic_random":
        if seed is not None:
            rng = np.random.default_rng(seed)
        else:
            rng = np.random.default_rng()
        cos_start = 1.0 # cos(0°), starting point for isotropic random sampling
        cos_end = float(np.cos(np.radians(max_deg))) # cos(max_deg), ending point for isotropic random sampling
        cos_vals = rng.uniform(cos_start, cos_end, n_inc) # generate n_inc random values in cos(i) for the isotropic random sampling
        cos_vals_clipped = np.clip(cos_vals, -1.0, 1.0) # clip the random cos(i) values to the valid range [-1, 1]
        vals = np.degrees(np.arccos(cos_vals_clipped)) # convert the random cos(i) values to inclination angles in degrees
    else:
        raise ValueError("Unknown mode: {}".format(mode))

    return np.asarray(vals, dtype=float) | units.deg

In [83]:
inclinations = get_inclinations(mode="kozai_regimes", n_inc=3)

In [84]:
def outer_orbit_energy_and_ecc(particles, G=constants.G):
    """Return (E_out, ecc_out, a_out) for Black Hole 2 relative to the inner binary (0:2)."""
    inner = particles[0:2]
    bh2 = particles[2]
    m_inner = inner.mass.sum()
    m_outer = bh2.mass
    mu = G * (m_inner + m_outer) #standard gravitational parameter for the outer orbit

    #bh2's position and velocity relative to the inner binary's center of mass
    r_vector = bh2.position - inner.center_of_mass() 
    v_vector = bh2.velocity - inner.center_of_mass_velocity()
    r = r_vector.length()
    v2 = v_vector.dot(v_vector)

    #orbital energy
    eps = 0.5 * v2 - mu / r #specific orbital energy of the outer orbit
    e_out_energy = (m_inner * m_outer / (m_inner + m_outer)) * eps #orbital energy of the outer orbit

    #eccentricity of the outer orbit (vector magnitude)
    e_vector = ((v2 - mu / r) * r_vector - r_vector.dot(v_vector) * v_vector) / mu 
    e_ecc_sq = e_vector.dot(e_vector)
    e_ecc_sq_val = float(e_ecc_sq.value_in(units.none) if hasattr(e_ecc_sq, "unit") else e_ecc_sq)
    ecc_out = float(np.sqrt(max(0.0, e_ecc_sq_val)))

    #semi-major axis (positive when eps < 0)
    a_out = -mu / (2.0 * eps)

    return e_out_energy, ecc_out, a_out

In [85]:
m_inner = (1 + 1 / 70) | MBlackhole
m_outer = (90 / 70) | MBlackhole
red_mass = (m_inner * m_outer) / (m_inner + m_outer)
initial_separation = 500 | units.AU

E_vals = [-1e36, -5e35, -1e35] | units.J #to start
L_vals = [3e45, 5e45, 8e45] | (units.kg * units.m**2 / units.s) #to start

eccentricities = []
phases_out = []
for E_val, L_val in zip(E_vals, L_vals):
    e_val, _, phase_val = get_orb_params(m_inner, m_outer, E_val, L_val, initial_separation)
    eccentricities.append(e_val)
    phases_out.append(phase_val)

phases_out = phases_out | units.deg

In [87]:
import sys
import traceback
import matplotlib.pyplot as plt

print("--- DIAGNOSTIC CHECK ---", flush=True)
print(f"eccentricities length: {len(eccentricities)}", flush=True)
print(f"semimajor_axes length: {len(semimajor_axes)}", flush=True)
print("------------------------\n", flush=True)

min_bound_steps = 3 
all_results = []
try:
    for idx, e_out in enumerate(eccentricities):
        for jdx, axis in enumerate(semimajor_axes):
            for kdx, inc_out in enumerate(inclinations):
                for ldx, phase_out in enumerate(phases_out):
                    axis_with_unit = axis if hasattr(axis, 'unit') else axis | units.AU
                    star_both_blackholes = initial_conditions_blackhole_star_system(
                        eccentricity_out=e_out,
                        eccentricity_star=0.0,
                        semimajor_axis_star=axis_with_unit,
                        inclination_out=inc_out,
                        true_anomaly_out=phase_out,
                        true_anomaly_star=0.0 | units.deg,
                    )
                    
                    a = [] | units.AU
                    e = []

                    results = []

                    orbit_inner = get_orbital_elements_from_binary(star_both_blackholes[0:2], G = constants.G)
                    print(f'black hole 1 and star orbit: {orbit_inner}')
                    a.append(orbit_inner[2])
                    e.append(orbit_inner[3])

                    combined_inner = Particles(1) 
                    combined_inner.mass = star_both_blackholes[0:2].mass.sum()
                    combined_inner.position = star_both_blackholes[0:2].center_of_mass()
                    combined_inner.velocity = star_both_blackholes[0:2].center_of_mass_velocity()

                    outer_system = Particles() 
                    outer_system.add_particle(combined_inner[0])
                    outer_system.add_particle(star_both_blackholes[2])
                    
                    orbit_outer = get_orbital_elements_from_binary(outer_system, G = constants.G)
                    print(f'black hole 1/star and black hole 2 orbit: {orbit_outer}')
                    a.append(orbit_outer[2])
                    e.append(orbit_outer[3])

                    #scatter(star_both_blackholes.x, star_both_blackholes.y, s=2+300*np.sqrt(star_both_blackholes.mass/star_both_blackholes.mass.sum()), c='b', marker="*")
                    #scatter(a, e)

                    # NOTE: The star is a point particle here. Replace 1 RSun with its
                    # stellar-model radius. This trigger is geometric contact, not a merger.
                    star_both_blackholes[0].radius = 0 | units.RSun
                    star_both_blackholes[1].radius = 1 | units.RSun
                    star_both_blackholes[2].radius = 0 | units.RSun

                    converter = nbody_system.nbody_to_si(star_both_blackholes.mass.sum(), axis_with_unit)
                    gravity = Ph4(converter)
                    try:
                        gravity.parameters.timestep_parameter = 0.01
                        gravity.particles.add_particles(star_both_blackholes)

                        # Ph4 stops evolve_model when the configured particle radii overlap.
                        collision_detection = gravity.stopping_conditions.collision_detection
                        if not collision_detection.is_supported():
                            gravity.stop()
                            raise RuntimeError("This Ph4 build does not support collision_detection.")
                        collision_detection.enable()
                        #print(f'gravity.particles: {gravity.particles}')
                        #print(f'bodies (star and bh1 and bh2): {star_both_blackholes}')

                        channel_from_gravity = gravity.particles.new_channel_to(star_both_blackholes)
                        channel_from_gravity.copy()
                        #print(f'updated bodies (star and bh1 and bh2): {star_both_blackholes}')

                        Porb = semimajor_axis_to_orbital_period(a[0], star_both_blackholes[0:2].mass.sum())
                        #print(f'Orbital period of black hole 1 and star: {Porb}')

                        Porb2 = semimajor_axis_to_orbital_period(a[1], star_both_blackholes[0:2].mass.sum() + star_both_blackholes[2].mass)
                        #print(f'Orbital period of inner binary and black hole 2: {Porb2}')

                        Ek0 = star_both_blackholes.kinetic_energy()
                        Ep0 = star_both_blackholes.potential_energy()
                        Etot0 = Ek0 + Ep0

                        # NOTE: Porb2 is a suitable end time only for a bound outer orbit.
                        # An unbound flyby needs a separately chosen end time.
                        a_out_au = a[1].value_in(units.AU)

                        if a_out_au > 0.0:
                            Porb2 = semimajor_axis_to_orbital_period(
                                a[1], star_both_blackholes[0:2].mass.sum() + star_both_blackholes[2].mass
                            )
                            end_time = Porb2.as_quantity_in(units.yr)
                        else:
                            r_out0_m = (star_both_blackholes[2].position - star_both_blackholes[0:2].center_of_mass()).length().value_in(units.m)
                            v_out0_mps = max(
                                (star_both_blackholes[2].velocity - star_both_blackholes[0:2].center_of_mass_velocity()).length().value_in(units.m / units.s),
                                1.0,
                            )
                            crossing_time_s = r_out0_m / v_out0_mps
                            end_time = (25.0 * crossing_time_s) | units.s
                            end_time = end_time.as_quantity_in(units.yr)

                        time = 0.0 | units.yr
                        end_time_yr = float(end_time.value_in(units.yr))
                        dt_yr = float((Porb / 250).value_in(units.yr))
                        if np.isfinite(end_time_yr) and end_time_yr > 0.0:
                            dt_yr = min(dt_yr, end_time_yr / 40.0)
                        dt = dt_yr | units.yr
                        index = 0

                        #tracking capture of bh2
                        bound_steps = 0
                        captured = False
                        capture_time = None
                        capture_ecc_out = np.nan
                        capture_a_out_AU = np.nan
                        min_e_out_J = np.inf
                        min_ecc_out = np.inf

                        while gravity.model_time < end_time:
                            time = min(gravity.model_time + dt, end_time)
                            gravity.evolve_model(time)
                            channel_from_gravity.copy()
                            
                            if collision_detection.is_set():
                            # Stopping-condition columns are aligned pair-by-pair.
                                for particle0, particle1 in zip(
                                    collision_detection.particles(0),
                                    collision_detection.particles(1),
                                ):
                                    separation = (particle0.position - particle1.position).length()
                                    print(
                                        f"Surface-contact trigger at {gravity.model_time.in_(units.yr)}: "
                                        f"{particle0.name} and {particle1.name}; "
                                        f"separation={separation.in_(units.RSun)}"
                                    )
                                    # This does not model hydrodynamics, capture, or a merger.
                                    break

                            Ek = star_both_blackholes.kinetic_energy()
                            Ep = star_both_blackholes.potential_energy()
                            Etot = Ek + Ep

                            #models capture of black hole 2 by the inner binary
                            #checks if black hole 2 is bound to black hole 1 and star binary
                            e_out_energy, ecc_out, a_out = outer_orbit_energy_and_ecc(star_both_blackholes, G=constants.G)
                            e_out_val_J = e_out_energy.value_in(units.J)
                            if np.isfinite(e_out_val_J):
                                min_e_out_J = min(min_e_out_J, e_out_val_J)
                            if np.isfinite(ecc_out):
                                min_ecc_out = min(min_ecc_out, ecc_out)
                            is_bound_step = (
                                (e_out_energy < 0.0 | units.J)
                                and (Etot < 0.0 | units.J)
                                and (0.0 <= ecc_out < 1.0)
                            )
                            if is_bound_step:
                                bound_steps += 1
                                if (not captured) and (bound_steps >= min_bound_steps):
                                    captured = True
                                    capture_time = gravity.model_time - (min_bound_steps - 1) * dt
                                    capture_ecc_out = ecc_out
                                    capture_a_out_AU = a_out.value_in(units.AU)
                            else:
                                bound_steps = 0
                            
                            final_inner = get_orbital_elements_from_binary(star_both_blackholes[0:2], G=constants.G)
                            results.append({
                                "e_out": e_out,
                                "axis_star_AU": axis_with_unit.value_in(units.AU),
                                "inc_out_deg": inc_out.value_in(units.deg),
                                "phase_out_deg": phase_out.value_in(units.deg),
                                "final_dE": ((Etot - Etot0) / Etot0),
                                "final_a_inner_AU": final_inner[2].value_in(units.AU),
                                "final_e_inner": final_inner[3],
                                "captured": captured,
                                "capture_time_yr": capture_time.value_in(units.yr) if capture_time is not None else np.nan,
                                "ecc_out_now": ecc_out,
                                "a_out_now_AU": a_out.value_in(units.AU) if e_out_energy < 0.0 | units.J else np.nan,
                            })

                            #for r in results:
                            #    print(
                            #        f"e_out={r['e_out']:.2f} | a_star={r['axis_star_AU']:.1f} AU | "
                            #        f"inc={r['inc_out_deg']:.0f} deg | phase={r['phase_out_deg']:.0f} deg | "
                            #        f"dE={r['final_dE']:.2e} | a_final={r['final_a_inner_AU']:.2f} AU | "
                            #        f"e_final={r['final_e_inner']:.3f}"
                            #    )

                            index += 1
                            if index % 10 == 0:  # save every 10th step to avoid huge image counts
                                plt.figure(figsize=(5, 5))
                                plt.xlim(-500, 500)
                                plt.ylim(-500, 500)
                                plt.scatter(
                                    star_both_blackholes.x.value_in(units.AU),
                                    star_both_blackholes.y.value_in(units.AU),
                                    s=2 + 300 * np.sqrt(star_both_blackholes.mass / star_both_blackholes.mass.sum()),
                                )
                                filename = f"full_system_snapshot_e{idx}_a{jdx}_inc{kdx}_ph{ldx}_i{index:04}.jpg"
                                plt.savefig(filename)
                                plt.close()
                        all_results.extend(results)
                        if results:
                            last = results[-1]
                            print(
                                f"DONE | e_out={last['e_out']:.2f} | a_star={last['axis_star_AU']:.1f} AU | "
                                f"captured={last['captured']} | t_cap_yr={last['capture_time_yr']:.2f} | "
                                f"min_e_out={min_e_out_J:.2e} J | min_ecc={min_ecc_out:.4f} | "
                                f"dE={last['final_dE']:.2e} | a_final={last['final_a_inner_AU']:.2f} AU | "
                                f"e_final={last['final_e_inner']:.3f}"
                            )
                        else:
                            print(f"SKIPPED/EMPTY | e_out={e_out:.2f} | a_star={axis_with_unit.value_in(units.AU):.1f} AU | end_time={end_time}")
                    except Exception:
                        traceback.print_exc()
                        raise
                    finally:
                        gravity.stop()
except Exception:
    traceback.print_exc()
    raise

--- DIAGNOSTIC CHECK ---
eccentricities length: 3
semimajor_axes length: 3
------------------------

black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<29.9999999944 149597870691.0 * m>, 0.0, quantity<3.14159265359 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<3.14159265359 rad>)
black hole 1/star and black hole 2 orbit: (quantity<1.01428571429 1.392244e+32 * kg>, quantity<1.28571428571 1.392244e+32 * kg>, quantity<1.09925134559e-05 149597870691.0 * m>, 0.9999989061911273, quantity<0.0548304837444 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<-6.00342248376e-08 rad>)
DONE | e_out=1.00 | a_star=30.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=-4.91e-16 | a_final=30.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<29.9999999944 149597870691.0 * m>, 0.0, quantity<3.14159265359 rad>, qua

/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


DONE | e_out=1.00 | a_star=30.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=-6.14e-16 | a_final=30.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<29.9999999944 149597870691.0 * m>, 0.0, quantity<3.14159265359 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<3.14159265359 rad>)
black hole 1/star and black hole 2 orbit: (quantity<1.01428571429 1.392244e+32 * kg>, quantity<1.28571428571 1.392244e+32 * kg>, quantity<1.06250970185e-05 149597870691.0 * m>, 0.999998868367178, quantity<0.0548292503323 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<-6.21088099634e-08 rad>)


/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


DONE | e_out=1.00 | a_star=30.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=-3.68e-16 | a_final=30.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<29.9999999944 149597870691.0 * m>, 0.0, quantity<3.14159265359 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<3.14159265359 rad>)
black hole 1/star and black hole 2 orbit: (quantity<1.01428571429 1.392244e+32 * kg>, quantity<1.28571428571 1.392244e+32 * kg>, quantity<1.09925134559e-05 149597870691.0 * m>, 0.9999989061911273, quantity<0.0548304837444 rad>, quantity<0.684719203002 rad>, quantity<-1.05109063536e-17 rad>, quantity<-6.00342248309e-08 rad>)


/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


... (27 outputs left out)

/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(


DONE | e_out=1.00 | a_star=50.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=6.14e-16 | a_final=50.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<100.000000004 149597870691.0 * m>, 0.0, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
black hole 1/star and black hole 2 orbit: (quantity<1.01428571429 1.392244e+32 * kg>, quantity<1.28571428571 1.392244e+32 * kg>, quantity<-7.04852166574e-07 149597870691.0 * m>, 1.000017058638597, quantity<0.054828988937 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<9.36250318959e-07 rad>)


/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/ext/orbital_elements.py:344: UserWarning: new_binary_from_orbital_elements is deprecated, use generate_binaries instead
  warnings.warn(
/Users/emorymurff/anaconda3/envs/amuse/lib/python3.14/site-packages/amuse/units/quantities.py:141: RuntimeWarning: invalid value encountered in scalar power
  return new_quantity(self.number**other, self.unit**other)


DONE | e_out=1.00 | a_star=100.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=0.00e+00 | a_final=100.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<100.000000004 149597870691.0 * m>, 0.0, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<0.0 rad>)
black hole 1/star and black hole 2 orbit: (quantity<1.01428571429 1.392244e+32 * kg>, quantity<1.28571428571 1.392244e+32 * kg>, quantity<-7.05477821719e-07 149597870691.0 * m>, 1.0000170435099622, quantity<0.0548284955877 rad>, quantity<0.0 rad>, quantity<0.0 rad>, quantity<9.35411558075e-07 rad>)
DONE | e_out=1.00 | a_star=100.0 AU | captured=False | t_cap_yr=nan | min_e_out=9.37e+53 J | min_ecc=2.2354 | dE=-1.23e-16 | a_final=100.00 AU | e_final=0.000
black hole 1 and star orbit: (quantity<1.0 1.392244e+32 * kg>, quantity<0.0142857142857 1.392244e+32 * kg>, quantity<100.000000004 149597870691.0 * m>

# unused code put in markdown cell

def initial_conditions_star_blackhole():
    # Define initial conditions for the black hole binary
    mass_blackhole_1 = 1 | MBlackhole
    mass_star = 1/70 | MBlackhole
    semimajor_axis_star = 30 | units.AU
    eccentricity_star = 0.0

    bodies = new_binary_from_orbital_elements(
        mass1 = mass_blackhole_1, 
        mass2 = mass_star, 
        semimajor_axis = semimajor_axis_star, 
        eccentricity = eccentricity_star, 
        G=constants.G) 
    bodies.name = ['Black Hole', 'Star']
    return bodies

def compute_energy(particles):
    return particles.kinetic_energy() + particles.potential_energy()

if __name__ == "__main__": #black hole and star binary system
    bodies = initial_conditions_star_blackhole()
    print(bodies)

    sma_star = []
    e_star = []

    star_orbit = get_orbital_elements_from_binary(bodies[0:2], constants.G)
    print(f'star orbit: {star_orbit}')

    sma_star.append(star_orbit[2])
    e_star.append(star_orbit[3])

    converter = nbody_system.nbody_to_si(bodies.mass.sum(), 30 | units.AU)
    gravity = Ph4(converter)
    gravity.particles.add_particles(bodies) #makes a copy of bodies in the nbody code called gravity
    channel_from_gravity_to_bodies = gravity.particles.new_channel_to(bodies) #makes a solid link from the memory in the python script to the memory in the nbody code

    Ek0 = bodies.kinetic_energy()
    Ep0 = bodies.potential_energy()
    Etot = Ek0 + Ep0

    Porb = semimajor_axis_to_orbital_period(sma_star[0], bodies[0:2].mass.sum())
    time = 0 | units.yr
    dt = Porb / 100
    #index = 0
    while gravity.model_time < Porb:
        time += dt
        gravity.evolve_model(time)
        channel_from_gravity_to_bodies.copy()
        Ek = bodies.kinetic_energy()
        Ep = bodies.potential_energy()
        #print(f'time = {time.in_(units.yr)}, dEtot = {(Ek+Ep-Etot)/Etot}')
        plt.xlim(-10, 10)
        plt.ylim(-10, 10)
        scatter(bodies.x, bodies.y, s = 2+300*np.sqrt(bodies.mass/bodies.mass.sum()), c = 'blue')
        #index += 1
        #filename = f'snapshot_i{index:04}.jpg' #makes images for the movie. has a lot of image outputs
        #plt.savefig(filename)
        #plt.close()




incoming_velocity_inf = 20 | units.kms #incoming velocity at infinty
impact_parameter = 1.5 | units.AU

energy = 1/2*incoming_velocity_inf**2
angular_momentum = incoming_velocity_inf * impact_parameter




mass_blackhole_1 = 1 | MBlackhole
mass_star = 1/70 | MBlackhole
mass_blackhole_2 = 90/70 | MBlackhole
Mtot = mass_star + mass_blackhole_1 + mass_blackhole_2

def eccentricity_and_pericenter(energy, angular_momentum, Mtot):
    eccentricity_blackhole = (1 + (2 * energy * angular_momentum**2)/((constants.G * Mtot)**2))**0.5
    pericenter_blackhole = angular_momentum**2 / ((constants.G * Mtot) * (1 + eccentricity_blackhole))
    return eccentricity_blackhole, pericenter_blackhole




eccentricity_blackhole_2, pericenter_blackhole_2 = eccentricity_and_pericenter(energy, angular_momentum, Mtot)
print(eccentricity_blackhole, pericenter_blackhole)





def initial_conditions_blackhole_binary():
    mass_blackhole_1 = 1 | MBlackhole
    mass_blackhole_2 = 90/70 | MBlackhole
    semimajor_axis_blackhole = 300 | units.AU
    eccentricity_blackhole = 0.0

    bodies = new_binary_from_orbital_elements(
        mass1 = mass_blackhole_1, 
        mass2 = mass_blackhole_2, 
        semimajor_axis = semimajor_axis_blackhole, 
        eccentricity = eccentricity_blackhole_2, 
        pericenter = pericenter_blackhole_2,
        G=constants.G) 
    bodies.name = ['Black Hole 1', 'Black Hole 2']
    return bodies

In [ ]:
# Building and checking an inbound hyperbolic three-body initial condition.
# Provisional values: v_inf=20 km/s, b=100 AU, r_init=50 b; inner orbit=30 AU.
import numpy as np
from amuse.datamodel import Particle
from amuse.ext.orbital_elements import new_binary_from_orbital_elements
from amuse.units import constants, units

black_hole_mass = 70 | units.MSun
star_mass = 1 | units.MSun
intruder_mass = 90 | units.MSun
inner_semimajor_axis = 30 | units.AU
inner_eccentricity = 0.0
v_inf = 20 | units.km / units.s
impact_parameter = 100 | units.AU
initial_separation = 50 * impact_parameter

inner_mass = black_hole_mass + star_mass
total_mass = inner_mass + intruder_mass
G = constants.G
h = impact_parameter * v_inf
p_out = h**2 / (G * total_mass)
e_out = np.sqrt(1 + (impact_parameter * v_inf**2 / (G * total_mass))**2)
r_peri = p_out / (1 + e_out)
cos_f = (p_out / initial_separation - 1) / e_out
if e_out <= 1:
    raise ValueError(f"Expected a hyperbolic approach (e > 1); got e={e_out}.")
if initial_separation <= r_peri or abs(cos_f) > 1:
    raise ValueError("Initial separation is not on the inbound branch of this hyperbola.")

true_anomaly = -np.arccos(cos_f)
radius = initial_separation.value_in(units.m)
r_relative = np.array([
    radius * np.cos(true_anomaly),
    radius * np.sin(true_anomaly),
    0.0,
]) | units.m
speed_scale = np.sqrt(G * total_mass / p_out)
v_relative = speed_scale * np.array([
    -np.sin(true_anomaly), e_out + np.cos(true_anomaly), 0.0,
])

inner_binary = new_binary_from_orbital_elements(
    mass1=black_hole_mass,
    mass2=star_mass,
    semimajor_axis=inner_semimajor_axis,
    eccentricity=inner_eccentricity,
    G=G,
)
inner_binary.name = ["BH 70 Msun", "Main-sequence star"]
inner_binary.position += -(intruder_mass / total_mass) * r_relative
inner_binary.velocity += -(intruder_mass / total_mass) * v_relative

intruder = Particle()
intruder.mass = intruder_mass
intruder.name = "Incoming BH 90 Msun"
intruder.position = (inner_mass / total_mass) * r_relative
intruder.velocity = (inner_mass / total_mass) * v_relative
initial_particles = inner_binary.copy()
initial_particles.add_particle(intruder)

print(f"Masses [Msun]: {black_hole_mass.value_in(units.MSun):.1f}, "
      f"{star_mass.value_in(units.MSun):.1f}, {intruder_mass.value_in(units.MSun):.1f}")
print(f"Incoming orbit: e={e_out:.6f}, "
      f"pericenter={r_peri.value_in(units.AU):.3f} AU, "
      f"r_init={initial_separation.value_in(units.AU):.1f} AU")
print(f"Created {len(initial_particles)} particles; "
      f"COM={initial_particles.center_of_mass().value_in(units.AU)} AU")

In [ ]:
# nIntegrate through the first close encounter and estimate whether
# the intruder is bound to the inner binary at the final snapshot.
from amuse.community.ph4.interface import Ph4
from amuse.units import nbody_system

step2_particles = initial_particles.copy()
step2_converter = nbody_system.nbody_to_si(total_mass, inner_semimajor_axis)
step2_gravity = Ph4(step2_converter)
step2_gravity.parameters.timestep_parameter = 0.01
try:
    step2_gravity.particles.add_particles(step2_particles)
    step2_channel = step2_gravity.particles.new_channel_to(step2_particles)
    step2_initial_energy = (
        step2_particles.kinetic_energy() + step2_particles.potential_energy()
    )

    # Approximate the inbound travel time by r_init/v_inf, then continue for
    # ten impact-parameter crossing times after pericenter.
    step2_end_time = (
        initial_separation / v_inf
        + 10 * impact_parameter / v_inf
    )
    step2_gravity.evolve_model(step2_end_time)
    step2_channel.copy()

    step2_final_energy = (
        step2_particles.kinetic_energy() + step2_particles.potential_energy()
    )
    step2_energy_drift = (
        (step2_final_energy - step2_initial_energy)
        / abs(step2_initial_energy)
    )

    # Treat the inner pair as a point mass for this instantaneous osculating
    # orbit diagnostic; it is not, on its own, proof of permanent capture.
    step2_inner = step2_particles[0:2]
    step2_intruder = step2_particles[2]
    step2_relative_position = (
        step2_intruder.position - step2_inner.center_of_mass()
    )
    step2_relative_velocity = (
        step2_intruder.velocity - step2_inner.center_of_mass_velocity()
    )
    step2_separation = step2_relative_position.length()
    step2_specific_orbital_energy = (
        0.5 * step2_relative_velocity.length()**2
        - constants.G * step2_inner.mass.sum() / step2_separation
    )

    print(f"Integrated to t={step2_end_time.value_in(units.yr):.2f} yr")
    print(f"Final total-energy drift: {step2_energy_drift:.3e}")
    print(
        "Final intruder separation from inner-binary COM: "
        f"{step2_separation.value_in(units.AU):.2f} AU"
    )
    print(
        "Final intruder relative speed: "
        f"{step2_relative_velocity.length().value_in(units.km / units.s):.3f} km/s"
    )
    print(
        "Final specific outer-orbit energy: "
        f"{step2_specific_orbital_energy.value_in(units.m**2 / units.s**2):.6e} m^2/s^2"
    )
    if step2_specific_orbital_energy.value_in(units.m**2 / units.s**2) < 0:
        print("Diagnostic: intruder is bound at this snapshot (permanent capture not established).")
    else:
        print("Diagnostic: intruder is unbound at this snapshot.")
finally:
    step2_gravity.stop()